# Create TROPESS CRIS-JPSS1 Daily Plots

## Overview
This notebook will allow you to download [TROPESS](https://tes.jpl.nasa.gov/tropess) CRIS-JPSS1 data and then use that data to create daily plots for a given date.  
The following suite of plots will be created from standard and summary data products (note that summary products for each species are not
always available):

| Species | Standard Product | Summary Product |
| :------ | :------ | :------ |
| Carbon Monoxide (CO) | TRPSDL2COCRS1FS | TRPSYL2COCRS1FS |
| Amonia (NH3) | TRPSDL2NH3CRS1FS | TRPSYL2NH3CRS1FS |
| Ozone (O3) | TRPSDL2O3CRS1FS | TRPSYL2O3CRS1FS |
| Methane (CH4) | TRPSDL2CH4CRS1FS | TRPSYL2CH4CRS1FS |
| Peroxyacetyl Nitrate (PAN) | TRPSDL2PANCRS1FS | TRPSYL2PANCRS1FS |
| Atmospheric Temperature (TATM) | TRPSDL2TATMCRS1FS | |
| Water (H2O) | TRPSDL2H2OCRS1FS | |
| Deuterated Water Vapor (HDO) | TRPSDL2HDOCRS1FS | |

In the table above, the "short name" of the data products is listed (e.g., "TRPSYL2O3CRS1FS").  Short names are assigned by a NASA DAAC (in this case, the GES-DISC) for each data products as a way to lookup or refer to a product with out using the product's full long name (e.g., "TROPESS CrIS-JPSS1 L2 Ozone for Forward Stream, Summary Product V1").  You'll see these short names in the code below when there are blocks looping through the different products.

## Requirements

This notebook uses NCDISC's [earthaccess](https://www.earthdata.nasa.gov/news/blog/earthaccess-earth-science-data-simplified) Python package to retrieve requested data.  It requires a [NASA Earthdata login](https://urs.earthdata.nasa.gov/).  Please make sure you have one before attempting to run this notebook.  Also make sure you've setup a `.netrc` file in your home directory with your NASA Earthata login information.  You can use the follow steps to generate that file, if needed:

```sh
cd ~
touch .netrc
echo "machine urs.earthdata.nasa.gov login uid_goes_here password password_goes_here" > .netrc
chmod 0600 .netrc
```

## Import Libraries 

Standard (i.e., available through pip) libraries `datetime`, `os`, and `glob` are imported below.  You will also need to clone or install the `tropessplots` library if you haven't already.  It is available at [https://github.com/NASA-TROPESS/tropessplots](https://github.com/NASA-TROPESS/tropessplots).

In [ ]:
import datetime as dt
import earthaccess
import glob
import os

from tropessplots.io.cris import read_l2summary, read_l2standard
from tropessplots.website_plots.cris import plot_daily_overview

%load_ext autoreload
%autoreload 2

## Setup Global Variables

Please edit the variables in the next block as needed.  We'll do some checks and conversions after those variables are
set to reformat the date so we can use it in later blocks and to create any directories that don't exist.  We also have a 
list of all the data products we want to plot.

In [ ]:
# The date you want to plot in YYYYMMDD format.
DATE = '20260602'

# Where you want to store the data you download.
DOWNLOAD_DIRECTORY = '/tmp/download'

# Where you want to store the plot outputs.
PLOT_DIRECTORY = '/tmp/download'

# A list of the species you want to plot.  All species are listed here
# by default.  Edits to this list (additions or subtractions) will be 
# reflected in the SHORT_NAME_LIST variable below.  
SPECIES_ARRAY = ['CO', 'NH3', 'O3', 'CH4', 'PAN', 'TATM', 'H2O', 'HDO']

# Data version.  As of this release, versions 1 (through May 2026) 
# and 2 (starts June 2026) are available
if dt.datetime.strptime(DATE, '%Y%m%d').date() <= dt.date(2026, 5, 31):
        VERSION = '1'
else:
        VERSION = '2'

In [ ]:
# The short names of the data products you want to plot.  These are automatically
# generated from SPECIES_ARRAY above.  Do not manually edit this list.
SHORT_NAME_LIST = []

if 'CO' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2COCRS1FS')
    SHORT_NAME_LIST.append('TRPSYL2COCRS1FS')
if 'NH3' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2NH3CRS1FS')
    SHORT_NAME_LIST.append('TRPSYL2NH3CRS1FS')
if 'O3' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2O3CRS1FS')
    SHORT_NAME_LIST.append('TRPSYL2O3CRS1FS')
if 'CH4' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2CH4CRS1FS')
    SHORT_NAME_LIST.append('TRPSYL2CH4CRS1FS')
if 'PAN' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2PANCRS1FS')
    SHORT_NAME_LIST.append('TRPSYL2PANCRS1FS')
if 'TATM' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2TATMCRS1FS')
if 'H2O' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2H2OCRS1FS')
if 'HDO' in SPECIES_ARRAY:
    SHORT_NAME_LIST.append('TRPSDL2HDOCRS1FS')

# Formatting dates for use later
date_object = dt.datetime.strptime(DATE, '%Y%m%d')
start_date_time = date_object.strftime('%Y-%m-%d 00:00:00')
end_date_time = date_object.strftime('%Y-%m-%d 23:59:59')

# Create directories if they don't already exist
os.makedirs(DOWNLOAD_DIRECTORY, exist_ok=True)
os.makedirs(PLOT_DIRECTORY, exist_ok=True)

## Search for and Download Data

Now that we are setup, we're going to start our work by downloading data from NASA Earthdata. We'll first search for files on the shot name, start and end dates, and version number.  Once the service returns with a list of matching files, we'll download them.

In [ ]:
for short_name in SHORT_NAME_LIST:
    earthaccess.login(strategy='netrc')
    results = earthaccess.search_data(count=-1, short_name=short_name, temporal=(start_date_time, end_date_time), version=VERSION)
    if len(results) == 0:
        continue
    print('Downloading %s' % short_name)
    earthaccess.download(results, DOWNLOAD_DIRECTORY)

## Plot the Data

Based on the `SHORT_NAMES_LIST` global variable, we'll crete an array of species names that we're going to plot.  We'll use the standard files for the plotting function, along with the summary files if they exist for that specific species.  (Note that a summary product does exist for HDO, but we won't use it in our plot.)

First, we'll find the standard and summary files on the local system.  Then we'll read the data with the readers we imported from the `tropessplots` library. Then, finally, we will plot the data, again with a plotting function from `tropessplots`.

In [ ]:
for species in SPECIES_ARRAY:    
    
    # Find files
    if species != 'TATM' and species != 'H2O' and species != 'HDO':
        l2summary_file = glob.glob(DOWNLOAD_DIRECTORY + '/*CrIS*Summary*' + species+'*' + DATE + '*.nc')
    else:
        l2summary_file = 'None'
    l2standard_file = glob.glob(DOWNLOAD_DIRECTORY + '/*CrIS*Standard*' + species + '*' + DATE + '*.nc')
    figure_file = PLOT_DIRECTORY + '/TROPESS_CrIS-JPSS1_' + species + '_' + DATE + '.png'
    
    # Read data
    if species != 'TATM' and species != 'H2O' and species != 'HDO':
        l2summary = read_l2summary(files=l2summary_file,
                                   verbose=0)
    else:
        l2summary = None
    l2standard = read_l2standard(files=l2standard_file,
                                 verbose=0)
    
    # Run plotting routine
    plot_daily_overview(l2summary=l2summary,
                        l2standard=l2standard,
                        file_out=figure_file)
    print('Produced plot %s' % figure_file)